In [ ]:
# pip install requests
# !pip install psycopg2
import requests   # for getting request from external API
import pandas as pd   # for convert loaded dataset into dataframe
import json   # load json file
from datetime import datetime   # datetime format
from uuid import uuid4   # for auto create unique id
import re   # regular expression for text comparison
import psycopg2   # for connecting dataframe with AWS RDS database 
from sqlalchemy import create_engine, text   # convert dataframe into postges database

In [220]:
# City of Melbourne token
headers = {
    "key_com" : "8272f3e3cf855ca3006bd9d38e135f06a0714adb8519bf503af526af"
}

In [221]:
# # Real-time: On-street parking bay sensors

# # On-street parking bay sensors
# onstreet_pbs_url = "https://data.melbourne.vic.gov.au/api/explore/v2.1/catalog/datasets/on-street-parking-bay-sensors/records"

# headers = {
#     "key_com" : "8272f3e3cf855ca3006bd9d38e135f06a0714adb8519bf503af526af"
# }
# # record limit per iteration
# record_limit = 100


# # loop interval: 2 min.
# loop_interval = 10

# # initialise current time - buffer
# current_time_buffer = (datetime.now() - timedelta(seconds=20)).isoformat()


# while True:
#     # initialise list for storing updated_data
#     updated_lst = []
#     offset = 0
#     while True:
#         condition = {"limit" : record_limit,
#                     "offset" : offset,
#                     "where" : f"lastupdated > '{current_time_buffer}'"}
        
#         # Get request
#         onstreet_pbs_response = requests.get(onstreet_pbs_url,
#                                             headers=headers,
#                                             params=condition)
#         # print test
#         test = onstreet_pbs_response.json()
#         print(test)

#         # HTTP breakout condition
#         res_status_cd = onstreet_pbs_response.status_code
#         if res_status_cd != 200:
#             if res_status_cd >= 500:
#                 print(f"Failed: {res_status_cd} Server Error")
#             else:
#                 print(f"Failed: {res_status_cd} Request Error", onstreet_pbs_response.text)
#             break

#         # Get response into record, csv format
#         record_json = onstreet_pbs_response.json()
#         record_onstreet_pbs = record_json.get("results", [])
#         print(f"Offset {offset}, fetched {len(record_onstreet_pbs)}")

#         # break condition if no record 
#         if not record_onstreet_pbs:
#             break
        
#         # append new record into list
#         updated_lst.extend(record_onstreet_pbs)

#         # update offest with record_limit
#         offset += record_limit

#     if updated_lst:
#         # convert json into pandas dataframe for data mapping
#         df_onstreet_pbs = pd.DataFrame(updated_lst)
#         print(f"{len(df_onstreet_pbs)} records updated.")

#         # update current time
#         current_time_buffer = max(pd.to_datetime(df_onstreet_pbs["lastupdated"]))
    
#     else:
#         print("No record updates.")
    
#     # loop report
#     time.sleep(loop_interval)

# print(df_onstreet_pbs)

In [ ]:
# 1. (Real-time: Get 100 records) On-street parking bay sensors
onstreet_pbs_url = "https://data.melbourne.vic.gov.au/api/explore/v2.1/catalog/datasets/on-street-parking-bay-sensors/records?limit=100"

# Get request
onstreet_pbs_response = requests.get(onstreet_pbs_url, headers=headers)

# Output
onstreet_pbs_data = onstreet_pbs_response.json()

In [ ]:
# extract record list
record_onstreet_pbs = onstreet_pbs_data["results"]

# convert json to dataframe
df_onstreet_pbs = pd.json_normalize(record_onstreet_pbs)

# rename column header
df_onstreet_pbs = df_onstreet_pbs.rename(
    columns={"zone_number" : "parkingzone",
             "status_description" : "status_desc", 
             "location.lon" : "longitude",
             "location.lat" : "latitude"}
    )

# convert datetime columns
df_onstreet_pbs["lastupdated"] = pd.to_datetime(df_onstreet_pbs["lastupdated"])
# df_onstreet_pbs["lastupdated"] = df_onstreet_pbs["lastupdated"].dt.strftime("%H:%M:%S")

df_onstreet_pbs["status_timestamp"] = pd.to_datetime(df_onstreet_pbs["status_timestamp"])
# df_onstreet_pbs["status_timestamp"] = df_onstreet_pbs["status_timestamp"].dt.strftime("%H:%M:%S")
df_onstreet_pbs["parking_date"] = df_onstreet_pbs["status_timestamp"].dt.date
df_onstreet_pbs["parking_time"] = df_onstreet_pbs["status_timestamp"].dt.time

df_onstreet_pbs["parkingzone"] = df_onstreet_pbs["parkingzone"].astype("Int64")


def parking_status(x):
    """
    Function convert parking status.
    Parameter: x is parking bay sensor record from status_description
    Returns:
    1 if parking is available (Unoccupied),
    0 if parking is not available (Present)
    """
    if x["status_desc"] == "Unoccupied":
        return True
    else:
        return False

df_onstreet_pbs["is_available"] = df_onstreet_pbs.apply(parking_status, axis=1)

# output
# print(df_onstreet_pbs.head())
# print(df_onstreet_pbs.dtypes)

# For SQL create table: PARKING_BAY_SENSOR (pk = kerbsideid, fk = parkingzone)
tb_parking_bay_sensor = df_onstreet_pbs[["kerbsideid", "lastupdated", "parking_date", "parking_time", "is_available", "parkingzone"]]

In [224]:
# # 2. Sign plates located in each parking zone
# sign_plates_loc_url = "https://data.melbourne.vic.gov.au/api/explore/v2.1/catalog/datasets/sign-plates-located-in-each-parking-zone/records?limit=100"

# # Get request
# sign_plates_loc_response = requests.get(sign_plates_loc_url, headers=headers)

# # Output
# sign_plates_loc_data = sign_plates_loc_response.json()

In [225]:
# # extract record list
# record_sign_plates_loc = sign_plates_loc_data["results"]

# # convert json to dataframe
# df_sign_plates_loc = pd.json_normalize(record_sign_plates_loc)

# # convert datetime columns
# df_sign_plates_loc["time_restrictions_start"] = pd.to_datetime(df_sign_plates_loc["time_restrictions_start"])
# df_sign_plates_loc["time_restrictions_start"] = df_sign_plates_loc["time_restrictions_start"].dt.time

# df_sign_plates_loc["time_restrictions_finish"] = pd.to_datetime(df_sign_plates_loc["time_restrictions_finish"])
# df_sign_plates_loc["time_restrictions_finish"] = df_sign_plates_loc["time_restrictions_finish"].dt.time

# # for checking
# print(df_sign_plates_loc)

# print(df_sign_plates_loc["restriction_display"].unique())

# # # Day of week dict {day:idx}
# # day_dict = {
# #     ""
# # }

In [226]:
# 2. (Static record) Sign plates located in each parking zone
with open("Dataset/sign-plates-located-in-each-parking-zone.json", "r") as file:
    data_sign_plates = json.load(file)

# convert json to dataframe
df_sign_plates = pd.json_normalize(data_sign_plates)

# format column data type
df_sign_plates["time_restrictions_start"] = pd.to_datetime(df_sign_plates["time_restrictions_start"], format="%H:%M:%S")
df_sign_plates["time_restrictions_start"] = df_sign_plates["time_restrictions_start"].dt.time

df_sign_plates["time_restrictions_finish"] = pd.to_datetime(df_sign_plates["time_restrictions_finish"], format="%H:%M:%S")
df_sign_plates["time_restrictions_finish"] = df_sign_plates["time_restrictions_finish"].dt.time
# output
# print(df_sign_plates.head())
# print(df_sign_plates.dtypes)

# For SQL create table: SIGN_PLATE (pk = (parkingzone, restruction_days), fk = parkingzone)
tb_sign_plates = (df_sign_plates
                  .dropna(subset=["parkingzone","restriction_days"])
                  .assign(parkingzone=pd.to_numeric(df_sign_plates["parkingzone"], errors="coerce"))
                  .astype({"parkingzone":"int64"})
                  .drop_duplicates(subset=["parkingzone","restriction_days"]))

In [228]:
# # 3. Parking zone linked to street segments
# parking_zone_url = "https://data.melbourne.vic.gov.au/api/explore/v2.1/catalog/datasets/parking-zones-linked-to-street-segments/records?limit=100"

# # Get request
# parking_zone_response = requests.get(parking_zone_url, headers=headers)

# # Output
# parking_zone_data = parking_zone_response.json()

In [229]:
# # extract record list
# record_parking_zone = parking_zone_data["results"]

# # convert json to dataframe
# record_parking_zone = pd.json_normalize(record_parking_zone)

# print(record_parking_zone)
# print(record_parking_zone.dtypes)

In [ ]:
# 3. (Static record) Parking zone linked to street segments
with open("Dataset/parking-zones-linked-to-street-segments.json", "r") as file:
    data_parking_zones = json.load(file)

df_parking_zones = pd.json_normalize(data_parking_zones)
# print(df_parking_zones)

zones_all = pd.concat(
    [
        df_parking_zones["parkingzone"],   # segments
        df_sign_plates["parkingzone"],     # sign plates
        df_onstreet_pbs["parkingzone"],    # sensors (real-time)
    ],
    ignore_index=True
)

# For SQL create table: PARKING_ZONE (pk = parkingzone)
tb_parking_zones = pd.to_numeric(df_parking_zones["parkingzone"]).dropna().drop_duplicates().to_frame("parkingzone")

     parkingzone         onstreet            streetfrom            streetto  \
0           7000      Poplar Road       Upfield Railway      Kendall Avenue   
1           7031  Cardigan Street    Argyle Place North      Grattan Street   
2           7068     Lygon Street        Faraday Street        Elgin Street   
3           7067     Lygon Street         Pelham Street  Argyle Place North   
4           7118  Swanston Street  Lincoln Square North      Grattan Street   
..           ...              ...                   ...                 ...   
793         7963   Rosslyn Street         Howard Street         King Street   
794         7950    Railway Place        Stanley Street        Roden Street   
795         7968   Spencer Street       La Trobe Street     Jeffcott Street   
796         7975   Stanley Street           King Street      Spencer Street   
797         7995   William Street          Capel Street      Rosslyn Street   

     segment_id  
0         22405  
1         20512

In [231]:
# 4. (Static record) Parking zone linked to street segments

# For SQL create table: PARKING_ZONE_SEGMENT (pk = (parkingzone,segment_id), fk = parkingzone,segment_id)
tb_parking_zone_segment = (df_parking_zones[["parkingzone", "segment_id"]]
                           .assign(
                               parkingzone=pd.to_numeric(df_parking_zones["parkingzone"], errors="coerce"),
                               segment_id=pd.to_numeric(df_parking_zones["segment_id"], errors="coerce"))
                               .dropna(subset=["parkingzone", "segment_id"])
                               .astype({"parkingzone":"int64","segment_id":"int64"})
                               .drop_duplicates(subset=["parkingzone","segment_id"]))

In [232]:
# 5. (Static record) Parking zone linked to street segments

# For SQL create table: PARKING_SEGMENT (pk = segment_id)
tb_parking_segment = (df_parking_zones[["segment_id", "onstreet", "streetfrom", "streetto"]]
                      .dropna(subset=["segment_id"])
                      .drop_duplicates(subset=["segment_id"]))

In [233]:
# 6. (static) On-street parking bays
with open("Dataset/on-street-parking-bays.json", "r") as file:
    data_parking_bays = json.load(file)

# convert json to dataframe
df_parking_bays = pd.json_normalize(data_parking_bays)

# rename column
df_parking_bays = df_parking_bays.rename(columns={"roadsegmentid" : "segment_id",
                                                  "roadsegmentdescription" : "segment_desc"})

# format column data type
df_parking_bays["kerbsideid"] = pd.to_numeric(df_parking_bays["kerbsideid"], errors="coerce")

df_parking_bays["lastupdated"] = pd.to_datetime(df_parking_bays["lastupdated"], format="%Y-%m-%d")

# filter necessary column
df_parking_bays = df_parking_bays.drop(columns=["location.lon","location.lat"])


# print(df_parking_bays.head())
# print(df_parking_bays.dtypes)

# For SQL create table: PARKING_BAY (pk = kerbsideid, fk = segment_id)
tb_parking_bays = df_parking_bays[["kerbsideid", "segment_id", "segment_desc", "latitude", "longitude"]]

In [234]:
# connect dataframe with AWS RDS database

endpoint = "ta43-onboarding.c5kcsm8im4cz.ap-southeast-2.rds.amazonaws.com"
database = "postgres"  # or your actual DB
username = "postgres"
password = "TA43Onboarding"

rds_engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{endpoint}:5432/{database}?sslmode=require"
)

# check connection
with rds_engine.connect() as conn:
    print(conn.execute(text("SELECT now()")).scalar())

2025-08-11 05:27:53.217773+00:00


In [235]:
# load dataframe into postgres sql: setup

load_schema = "public"

# helper: check column name using regex
def _qident(name: str) -> str:
    """Check SQL identifier (column_name format)."""
    if not re.match(r'^[A-Za-z_][A-Za-z0-9_]*$', name):
        # raise value error if pk is missing
        raise ValueError(f"Unsafe identifier/column name: {name}")
    return f'"{name}"'

# helper: drop NA, convert dataframe to strings, and convert it to set for sorting
def to_str_set(s: pd.Series) -> set:
    return set(s.dropna().astype(str))

# load pandas dataframe into postgres table using staging table (temp sql table)
def upsert_do_nothing(df, target_table, pk_cols, conn, schema=load_schema, chunksize=10_000, cast_map=None):
    """
    1. target table columns (ordered)
    2. check if Primary Key columns exist on target table column
    3. load columns that exist on target (keeps order)
    4. create temp table name
    5. write dataframe to staging table (sql format)
    6. build insert and select statement, then run upsert
    7. run insert (when there is conflict, do nothing)
    Parameters:
        df = dataframe
        target_table = table to compare
        pk_cols = table primary key
        conn = connect engine to ASW RDS database
        schema = load data schema
        chunksize = load batch size
        cast_map = maping temp column with actual column
    Return:
        insert dataframe into postgres table
    """
    # 1. target table columns (ordered)
    cols_rs = conn.execute(text("""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = :schema AND table_name = :table
        ORDER BY ordinal_position
    """), {"schema": schema, "table": target_table})
    # temp table for comparison
    target_cols = []    # initialise target column empty list
    for row in cols_rs:
        target_cols.append(row[0])
        
    # 2. check if Primary Key columns exist on target table column
    missing_pks = []    # initialise missing PK empty list
    for column in pk_cols:
        if column not in target_cols:
            missing_pks.append(column)
    # raise value error if pk is missing
    if missing_pks:
        raise ValueError(f"PK columns not found in {target_table}: {missing_pks}")

    # 3. load columns that exist on target (keeps order)
    col_in_df = set(df.columns)     # initialise column in dataframe as set to keep its order
    cols_to_load = [c for c in target_cols if c in col_in_df]
    if not cols_to_load:
        # raise value error when no columns overlaps
        raise ValueError(f"No overlapping columns between dataframe and {target_table}.")

    # 4. create temp table name
    temp_unique_id = uuid4().hex[:8]    # using 8 random hex chars 
    stage_table = f"_stg_{target_table}_{temp_unique_id}"

    # 5. write dataframe to staging table (sql format)
    df.to_sql(
        stage_table,
        conn,
        schema=schema,
        if_exists="replace",
        index=False,
        method="multi",
        chunksize=chunksize,
    )

    # 6. build insert and select statement, then run upsert
    col_list = ", ".join(_qident(c) for c in cols_to_load)
    pk_list  = ", ".join(_qident(c) for c in pk_cols)
    full_target = f'{_qident(schema)}.{_qident(target_table)}'
    full_stage  = f'{_qident(schema)}.{_qident(stage_table)}'

    sel_exprs = []  # initialise empty list
    for column in cols_to_load:
        if cast_map and column in cast_map:
            # append if cast_map exist and has entry for that column
            sel_exprs.append(f'{_qident(column)}::{cast_map[column]}')
        else:
            sel_exprs.append(_qident(column))
    # join sel_list, sel_expres
    sel_list = ", ".join(sel_exprs)

    # 7. run insert (when there is conflict, do nothing)
    conn.execute(text(f"""
        INSERT INTO {full_target} ({col_list})
        SELECT {sel_list}
        FROM {full_stage}
        ON CONFLICT ({pk_list}) DO NOTHING
    """))

    # 8. drop temp table
    conn.execute(text(f"DROP TABLE {full_stage}"))


# load dataframe into postgres table

with rds_engine.begin() as conn:
    # 1. load parents table: PARKING_ZONE, PARKING_SEGMENT
    # 1.1 PARKING_ZONE (pk = parkingzone)
    upsert_do_nothing(tb_parking_zones, "parking_zone", ["parkingzone"], conn)

    # 1.2 PARKING_SEGMENT (pk = segment_id)
    upsert_do_nothing(tb_parking_segment, "parking_segment", ["segment_id"], conn)

    # 2. fetch current parent keys from database
    zones_key_sql = to_str_set(pd.read_sql_query("SELECT parkingzone FROM parking_zone", conn)["parkingzone"])
    segments_key_sql  = to_str_set(pd.read_sql_query("SELECT segment_id FROM parking_segment", conn)["segment_id"])

    # 3. filter children by existing parents (avoid foreign key violations)
    # 3.1 SIGN_PLATE (fk: parkingzone)
    sp_ok  = tb_sign_plates[tb_sign_plates["parkingzone"].astype(str).isin(zones_key_sql)].copy()
    
    # 3.2 PARKING_ZONE_SEGMENT (fk: parkingzone, segment_id)
    pzs_ok = tb_parking_zone_segment[
        tb_parking_zone_segment["parkingzone"].astype(str).isin(zones_key_sql) &
        tb_parking_zone_segment["segment_id"].astype(str).isin(segments_key_sql)].copy()

    # 3.3 PARKING_BAY (fk: segment_id)
    bays_ok = tb_parking_bays[tb_parking_bays["segment_id"].astype(str).isin(segments_key_sql)].copy()
    bays_ok = bays_ok[bays_ok["kerbsideid"].notna()]     # drop null PKs

    # 3.4 PARKING_BAY_SENSOR (fk: parkingzone)
    sens_ok = tb_parking_bay_sensor[
        tb_parking_bay_sensor["parkingzone"].notna() &
        tb_parking_bay_sensor["parkingzone"].astype(str).isin(zones_key_sql)].copy()

    # 4. upsert the filtered children
    # 4.1 SIGN_PLATE
    upsert_do_nothing(sp_ok, "sign_plate", ["parkingzone", "restriction_days"], conn, cast_map={"time_restrictions_start": "time", "time_restrictions_finish": "time",})
    
    # 4.2 PARKING_ZONE_SEGMENT
    upsert_do_nothing(pzs_ok, "parking_zone_segment", ["parkingzone", "segment_id"], conn)
    
    # 4.3 PARKING_BAY
    upsert_do_nothing(bays_ok, "parking_bay", ["kerbsideid"], conn)

    # 4.4 PARKING_BAY_SENSOR
    upsert_do_nothing(sens_ok, "parking_bay_sensor", ["kerbsideid"], conn)
